<a href="https://colab.research.google.com/github/CaesarGhazi/Flyrank/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CaesarGhazi/Flyrank/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

gsc_impressions and gsc_clicks are heavily right-skewed — a small number of pages carry most of the volume (consistent with the 99%/1% cluster split found later). ctr is near-zero for the majority of content, with a long thin tail of higher-performing pages. This heavy-tail shape is exactly why a median-based CTR threshold silently failed in earlier work — most values cluster near the floor.

In [2]:
import pandas as pd, numpy as np, os
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
HF_BASE = "hf://datasets/FlyRank/internship-warehouse"

fact_cols = ["report_date","client_hash_id","content_hash_id","gsc_data_available","ga4_data_available",
             "gsc_impressions","gsc_clicks","gsc_avg_position","ga4_engaged_sessions"]
panel_daily = pd.read_parquet(f"{HF_BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet",
                               columns=fact_cols, storage_options={"token": HF_TOKEN})
dim_content = pd.read_parquet(f"{HF_BASE}/dim_content.parquet",
                               columns=["client_hash_id","content_hash_id","content_type"],
                               storage_options={"token": HF_TOKEN})
for c in ["client_hash_id","content_hash_id"]:
    panel_daily[c] = panel_daily[c].astype("category"); dim_content[c] = dim_content[c].astype("category")
dim_content = dim_content.drop_duplicates(subset=["client_hash_id","content_hash_id"])
panel_daily = panel_daily.merge(dim_content, on=["client_hash_id","content_hash_id"], how="left")

content_level = panel_daily.groupby(["client_hash_id","content_hash_id"], observed=True).agg(
    gsc_impressions=("gsc_impressions","sum"), gsc_clicks=("gsc_clicks","sum"),
    gsc_avg_position=("gsc_avg_position","mean"), ga4_engaged_sessions=("ga4_engaged_sessions","sum"),
    content_type=("content_type","first")).reset_index()
print("Shape:", content_level.shape)

Shape: (331437, 7)


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
content_level["ctr"] = content_level["gsc_clicks"] / content_level["gsc_impressions"].replace(0, np.nan)
content_level["engagement_rate"] = content_level["ga4_engaged_sessions"] / content_level["gsc_clicks"].replace(0, np.nan)

print(content_level[["gsc_impressions","gsc_clicks","gsc_avg_position","ctr","engagement_rate"]].describe())

       gsc_impressions     gsc_clicks  gsc_avg_position            ctr  \
count    331437.000000  331437.000000     176738.000000  176738.000000   
mean        846.790156       2.479602         15.999277       0.004594   
std        4044.514753      19.651282         17.686260       0.037760   
min           0.000000       0.000000          0.000000       0.000000   
25%           0.000000       0.000000          5.001970       0.000000   
50%           2.000000       0.000000          8.505296       0.000000   
75%         216.000000       0.000000         20.369190       0.002158   
max      617124.000000    5668.000000        309.000000       1.000000   

       engagement_rate  
count     68837.000000  
mean          0.047674  
std           0.228590  
min           0.000000  
25%           0.000000  
50%           0.000000  
75%           0.000000  
max          34.000000  


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

Verdict: CONFIRMED — avg CTR drops from 0.0106 (top 3) to 0.0019 (21-100), across well-populated buckets.

Verdict: OPPOSITE — highest-volume quartile shows lower avg engagement (0.046) than lower-volume buckets (~0.058).

Verdict: MIXED — keyword articles average 1,009 impressions, compared with 138 for comparison articles and 16 for Feedly articles, but content type is only a proxy for age, so age itself is not directly confirmed.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Signal 1: CTR vs position (flag-linked, see Section 3)
position_bins = pd.cut(content_level["gsc_avg_position"], bins=[0,3,10,20,100,1000])
sig1 = content_level.groupby(position_bins, observed=True).agg(n=("ctr","count"), avg_ctr=("ctr","mean"))
print(sig1)

# Signal 2: volume vs engagement
volume_bins = pd.qcut(content_level["gsc_impressions"], q=4, duplicates="drop")
sig2 = content_level.groupby(volume_bins, observed=True).agg(n=("engagement_rate","count"), avg_engagement=("engagement_rate","mean"))
print(sig2)

# Signal 3: content age proxy (content_type) vs impressions
sig3 = content_level.groupby("content_type", observed=True).agg(n=("gsc_impressions","count"), avg_impr=("gsc_impressions","mean"))
print(sig3)


                      n   avg_ctr
gsc_avg_position                 
(0, 3]            16144  0.010589
(3, 10]           81988  0.004926
(10, 20]          32203  0.003211
(20, 100]         44867  0.001918
(100, 1000]         102  0.006127
                       n  avg_engagement
gsc_impressions                         
(-0.001, 2.0]        313        0.057508
(2.0, 216.0]        9182        0.057891
(216.0, 617124.0]  59342        0.046041
                         n     avg_impr
content_type                           
comparison article    3392   138.298349
feedly article       51163    16.212497
keyword article     276882  1008.946053


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

Yes, confirmed — the assumption behind the flag holds in this data

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Re-uses Signal 1 — the exact signal behind FlyRank's real CTR-fix flag logic
print(sig1)


                      n   avg_ctr
gsc_avg_position                 
(0, 3]            16144  0.010589
(3, 10]           81988  0.004926
(10, 20]          32203  0.003211
(20, 100]         44867  0.001918
(100, 1000]         102  0.006127


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

A content team can trust CTR-vs-position as a real signal for prioritizing metadata fixes — it's confirmed, not assumed. But they should not assume high-traffic content is automatically the most engaging (Signal 2 is OPPOSITE) — traffic volume alone is a weak basis for a "quick win" rule, and any threshold built on it (like a median) needs checking against the real distribution first, since heavy tails can silently break naive rules.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.